# LatComm: запуск из своего форка

Этот ноутбук клонирует **[SalavatFayzullin/LatComm](https://github.com/SalavatFayzullin/LatComm)**,
устанавливает среду и запускает `kaggle/run.py` из клонированного репозитория.
Перед первым запуском **закоммитьте и отправьте папку `kaggle/` в свой форк**.

В Kaggle включите **Internet** и **GPU T4 ×2** или **P100**.
Данные авторов загружаются автоматически с Hugging Face.
`PROFILE = "full"` использует полный набор; `"smoke"` проверяет всю цепочку на малой выборке.
Даже в smoke исходный архив Stage 1 (около 2,8 GB) скачивается целиком.

Новый запуск берёт `REPO_REF`. Для продолжения обучения используется SHA из сохранённого
`manifest.json`, чтобы обновление ветки main не изменило код посреди эксперимента.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"
REPO_REF = "main"                       # branch/tag for a new experiment
SOURCE_COMMIT = None                    # optional full SHA for a fixed version
REPO_DIR = "/kaggle/working/LatComm"     # source code from your fork
PROFILE = "full"                       # "smoke" for a small pipeline check
WORKDIR = f"/kaggle/working/latcomm-{PROFILE}"  # checkpoints/results, separate from source
PAIR = "qwen4b_to_qwen0.8b"             # alternative: qwen0.8b_to_qwen0.8b
ACTION = "all"                         # setup, prepare, stage1, encode, stage2, evaluate, all
NUM_GPUS = 0                           # 0 = all visible GPUs
SESSION_STEPS = 25                     # new updates per session, full schedule is retained
SESSION_HOURS = 8.0                    # graceful stop at the next optimizer boundary
BENCHMARK_MICROBATCHES = 0              # 3 = time only 3 microbatches; use a fresh WORKDIR
RESTORE = None                        # e.g. /kaggle/input/previous-output/latcomm-full
DATA_SOURCE = None                    # optional local copy of published data
DATA_DIR = f"/kaggle/temp/latcomm-data-{PROFILE}"
PACKET_CACHE = WORKDIR + "/packets"    # full cache can require tens of GB
SKIP_SETUP = False                    # True only if this session's environment is installed
BOOTSTRAP_FILE = Path("/kaggle/working/latcomm_bootstrap.py")

os.environ["HF_HOME"] = "/kaggle/temp/latcomm-hf"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass


In [ ]:
# Bootstrap only; training code will be loaded from your fork.
BOOTSTRAP_FILE.write_text('"""Clone the user\'s LatComm fork, install the environment, run kaggle/run.py.\n\nThis entry point uses only Python\'s standard library and git. It can be copied\ninto a Kaggle cell or used by the accompanying lightweight notebook.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nREPO_URL = "https://github.com/SalavatFayzullin/LatComm.git"\nREQUIRED = ("kaggle/run.py", "kaggle/runtime.py", "kaggle/data_utils.py", "latcomm/model.py")\n\n\ndef call(command, **kwargs):\n    print("+", " ".join(map(str, command)), flush=True)\n    return subprocess.run(list(map(str, command)), check=True, **kwargs)\n\n\ndef git(repo: Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(repo), *arguments], text=True).strip()\n\n\ndef canonical_url(value: str) -> str:\n    return value.rstrip("/").removesuffix(".git")\n\n\ndef pinned_commit(args) -> str | None:\n    """A resumed experiment uses its saved SHA even when main has advanced."""\n    pins = []\n    for directory in (args.workdir, args.restore):\n        if directory is None:\n            continue\n        path = Path(directory) / "manifest.json"\n        if args.restore == directory and not path.is_file():\n            raise FileNotFoundError(f"Restore directory has no manifest.json: {directory}")\n        if not path.is_file():\n            continue\n        state = json.loads(path.read_text(encoding="utf-8"))\n        if canonical_url(state["source_url"]) != canonical_url(args.repo_url):\n            raise ValueError("Saved run belongs to another repository; choose another workdir.")\n        pins.append(state["source_commit"])\n    if args.commit:\n        pins.append(args.commit)\n    if len(set(pins)) > 1:\n        raise ValueError("Requested commit differs from the saved run; use its original commit or a new workdir.")\n    commit = pins[0] if pins else None\n    if commit is not None and not re.fullmatch(r"[0-9a-fA-F]{40}", commit):\n        raise ValueError("A pinned commit must be a full 40-character Git SHA.")\n    return commit\n\n\ndef checkout(args) -> tuple[Path, str]:\n    repo = args.repo_dir.resolve()\n    pin = pinned_commit(args)\n    fresh = False\n    if not args.ref or args.ref.startswith("-") or args.repo_url.startswith("-"):\n        raise ValueError("Invalid repository URL or ref.")\n    if not (repo / ".git").exists():\n        if repo.exists() and any(repo.iterdir()):\n            raise RuntimeError(f"Refusing to overwrite a nonempty directory: {repo}")\n        repo.parent.mkdir(parents=True, exist_ok=True)\n        call(["git", "clone", "--no-checkout", "--", args.repo_url, repo])\n        fresh = True\n    if canonical_url(git(repo, "remote", "get-url", "origin")) != canonical_url(args.repo_url):\n        raise RuntimeError(f"{repo} belongs to another repository; choose another --repo-dir.")\n    # Do not reset or overwrite local modifications in an existing Kaggle session.\n    if not fresh and git(repo, "status", "--porcelain", "--untracked-files=no"):\n        raise RuntimeError(f"Checkout has modified tracked files: {repo}. Keep it or choose another --repo-dir.")\n    if pin:\n        available = subprocess.run(["git", "-C", str(repo), "cat-file", "-e", f"{pin}^{{commit}}"],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n        if available.returncode:\n            call(["git", "-C", repo, "fetch", "--no-tags", "origin", pin])\n        revision = pin\n    else:\n        call(["git", "-C", repo, "fetch", "--no-tags", "origin", args.ref])\n        revision = git(repo, "rev-parse", "FETCH_HEAD^{commit}")\n    call(["git", "-C", repo, "checkout", "--detach", revision])\n    commit = git(repo, "rev-parse", "HEAD")\n    missing = [name for name in REQUIRED if not (repo / name).is_file()]\n    if missing:\n        raise RuntimeError(\n            f"Fork revision {commit} is missing {\', \'.join(missing)}. "\n            "Commit and push the kaggle/ scripts to your fork, then select that branch or commit."\n        )\n    print(f"LatComm fork: {args.repo_url}\\nSource commit: {commit}", flush=True)\n    return repo, commit\n\n\ndef training_arguments(args):\n    command = [args.action, "--workdir", str(args.workdir), "--profile", args.profile,\n               "--pair", args.pair, "--gpus", str(args.gpus), "--microbatch", str(args.microbatch),\n               "--precision", args.precision, "--session-steps", str(args.session_steps),\n               "--session-hours", str(args.session_hours), "--checkpoint-every", str(args.checkpoint_every)]\n    for name in ("restore", "data_source", "data_dir", "packet_cache", "data_revision"):\n        value = getattr(args, name)\n        if value is not None:\n            command += ["--" + name.replace("_", "-"), str(value)]\n    if args.benchmark_microbatches:\n        command += ["--benchmark-microbatches", str(args.benchmark_microbatches)]\n    return command\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-url", default=REPO_URL)\n    parser.add_argument("--ref", default="main", help="Branch/tag used for a new experiment.")\n    parser.add_argument("--commit", help="Optional full SHA; saved runs automatically pin their original SHA.")\n    parser.add_argument("--repo-dir", type=Path, default=Path("/kaggle/working/LatComm"))\n    parser.add_argument("--workdir", type=Path)\n    parser.add_argument("--profile", choices=("full", "smoke"), default="full")\n    parser.add_argument("--pair", default="qwen4b_to_qwen0.8b")\n    parser.add_argument("--action", choices=("setup", "prepare", "stage1", "encode", "stage2", "evaluate", "all"), default="all")\n    parser.add_argument("--gpus", type=int, default=0)\n    parser.add_argument("--microbatch", type=int, default=1)\n    parser.add_argument("--precision", choices=("auto", "fp16", "bf16"), default="auto")\n    parser.add_argument("--session-steps", type=int, default=25)\n    parser.add_argument("--session-hours", type=float, default=8.0)\n    parser.add_argument("--checkpoint-every", type=int, default=5)\n    parser.add_argument("--benchmark-microbatches", type=int, default=0)\n    parser.add_argument("--restore", type=Path)\n    parser.add_argument("--data-source", type=Path)\n    parser.add_argument("--data-dir", type=Path)\n    parser.add_argument("--packet-cache", type=Path)\n    parser.add_argument("--data-revision")\n    parser.add_argument("--skip-setup", action="store_true", help="Reuse an environment already installed in this session.")\n    args = parser.parse_args(argv)\n    args.workdir = (args.workdir or Path(f"/kaggle/working/latcomm-{args.profile}")).resolve()\n    if args.restore:\n        args.restore = args.restore.resolve()\n    if args.gpus < 0 or args.microbatch < 1 or args.checkpoint_every < 1 or args.session_steps < 0 or args.session_hours < 0 or args.benchmark_microbatches < 0:\n        parser.error("GPU count and session limits must be nonnegative; microbatch/checkpoint interval must be positive.")\n    repo, commit = checkout(args)\n    script = repo / "kaggle/run.py"\n    env = dict(os.environ)\n    env["LATCOMM_SOURCE_COMMIT"] = commit\n    env["PYTHONPATH"] = os.pathsep.join([str(repo), str(repo / "kaggle"), env.get("PYTHONPATH", "")])\n    if not args.skip_setup or args.action == "setup":\n        call([sys.executable, script, "setup", "--workdir", args.workdir], cwd=repo, env=env)\n    if args.action == "setup":\n        return 0\n    command = [sys.executable, str(script), *training_arguments(args)]\n    print("+", " ".join(command), flush=True)\n    result = subprocess.run(command, cwd=repo, env=env)\n    if result.returncode == 75:\n        print("Session state saved. Resume with the same workdir or --restore in a later session.", flush=True)\n    return result.returncode\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding='utf-8')


## Клонирование, установка и запуск

Следующая ячейка клонирует форк, выводит фактический SHA, выполняет
`python kaggle/run.py setup`, затем выбранный `ACTION` из того же checkout.
Повторное клонирование репозитория авторов не выполняется.
Установка и обучение идут в отдельных Python-процессах.

PyTorch закреплён на **2.11.0 cu126** для T4/P100; Transformers — **5.15.1**.
На этих GPU LLM работают в FP16, обучаемый интерфейс — в FP32, пакеты хранятся в BF16.
Это сохраняет алгоритм и отличается по вычислительной точности от H100 авторов.


In [ ]:
command = [sys.executable, str(BOOTSTRAP_FILE),
           "--repo-url", REPO_URL, "--ref", REPO_REF, "--repo-dir", REPO_DIR,
           "--workdir", WORKDIR, "--profile", PROFILE, "--pair", PAIR,
           "--action", ACTION, "--gpus", str(NUM_GPUS),
           "--session-steps", str(SESSION_STEPS), "--session-hours", str(SESSION_HOURS),
           "--benchmark-microbatches", str(BENCHMARK_MICROBATCHES),
           "--data-dir", DATA_DIR, "--packet-cache", PACKET_CACHE]
if SOURCE_COMMIT:
    command += ["--commit", SOURCE_COMMIT]
if RESTORE:
    command += ["--restore", RESTORE]
if DATA_SOURCE:
    command += ["--data-source", DATA_SOURCE]
if SKIP_SETUP:
    command += ["--skip-setup"]
result = subprocess.run(command)
if result.returncode == 75:
    print("Сессия завершена штатно; сохраните Output и продолжите с RESTORE.")
elif result.returncode:
    raise RuntimeError(f"LatComm failed, exit code {result.returncode}. See logs above.")


## Продолжение и результаты

Код 75 означает штатную паузу: сохранены optimizer, scheduler, scaler, RNG и позиция в данных.
Для следующей сессии сохраните Output, подключите его через Input и укажите `RESTORE`
на прежний `WORKDIR`, содержащий `manifest.json` и `outputs`.
Число GPU, микробатч, профиль и пару моделей при точном продолжении сохраняйте.

`/kaggle/temp` не сохраняется; JSONL и модели можно скачать повторно с закреплённых ревизий.
Кэш пакетов сохраняется отдельно. Его полный объём может превышать лимит сохраняемого output Kaggle.
Готовый совместимый кэш можно подключить из `/kaggle/input/...` через `PACKET_CACHE`.
Полный эксперимент требует многих сессий; `SESSION_STEPS` ограничивает текущую сессию.


In [ ]:
import json
benchmark = Path(WORKDIR) / "outputs/stage1/benchmark.json"
metrics = Path(WORKDIR) / "results/metrics.json"
if benchmark.exists():
    print(json.dumps(json.loads(benchmark.read_text()), indent=2, ensure_ascii=False))
elif metrics.exists():
    print(json.dumps(json.loads(metrics.read_text()), indent=2, ensure_ascii=False))
else:
    print("Оценка ещё не завершена. Checkpoints:", str(Path(WORKDIR) / "outputs"))


Это запуск LatComm для Kaggle на T4 ×2 или P100 **из своего форка [SalavatFayzullin/LatComm](https://github.com/SalavatFayzullin/LatComm)**. Все модули и конфигурации берутся из одного checkout этого форка. Дополнительный runtime отвечает за память, FP16, сохранение состояния и продолжение между сессиями. Файлы `latcomm/bridge.py` и `latcomm/model.py` не переписаны.

Перед первым запуском закоммитьте и отправьте папку `kaggle/` в выбранную ветку форка. Например, из корня локального репозитория:

```bash
git add kaggle README.md
git commit -m "Add Kaggle launcher"
git push origin main
```

Импортируйте `kaggle/LatComm_Kaggle.ipynb` в Kaggle. Выберите **GPU T4 ×2** (предпочтительно) или **P100**, включите **Internet**, затем запускайте ячейки сверху вниз. Ноутбук содержит только bootstrap: он клонирует ваш форк в `/kaggle/working/LatComm`, запускает `python kaggle/run.py setup`, затем `python kaggle/run.py all` из того же репозитория. Установка и обучение выполняются в отдельных Python-процессах.

`REPO_REF = "main"` выбирает ветку для нового эксперимента. `SOURCE_COMMIT` позволяет указать конкретный полный SHA. Фактический коммит записывается в `manifest.json`; при повторном запуске или `RESTORE` bootstrap автоматически выбирает прежний SHA, даже если ветка main обновилась. Для эксперимента с обновлённым кодом выберите новый `WORKDIR`. Существующий checkout с изменёнными tracked-файлами автоматически не перезаписывается.

Данные уже опубликованы авторами: [huaweilin/LatComm](https://huggingface.co/datasets/huaweilin/LatComm). Загрузчик использует исходные сжатые JSONL из этого репозитория, сохраняя разбиения и формулировки вопросов.

| Этап | Данные | Train | Validation | Test |
| --- | --- | ---: | ---: | ---: |
| Stage 1 | WikiText-103 + FineWeb-Edu, восстановление текста | 2 000 000 | 15 043 | 14 983 |
| Stage 2 | 12 наборов вопросов с выбором ответа | 282 441 | 2 036 | 55 890 |

На Stage 2 входят CLOTH, Cosmos QA, HellaSwag, LogiQA, MCScript2.0, MuTual, OpenBookQA, QuALITY, ReClor, SciQ, SocialIQA и SWAG. RACE, DREAM и QuAIL — дополнительные эксперименты переноса из статьи; данный запуск оценивает опубликованный основной тестовый набор из 12 задач.

В настройках ноутбука:

- `PROFILE = "full"` использует все данные статьи. Эпохи: 1 для Stage 1, затем 3 для Stage 2. Основная пара: Qwen3.5-4B → Qwen3.5-0.8B.
- `PROFILE = "smoke"` — явно уменьшенная проверка всей цепочки на небольшом числе примеров. Её результаты нельзя сравнивать с таблицами статьи. Ноутбук использует отдельный `WORKDIR` для каждого профиля. Загрузка исходного архива Stage 1 (около 2,8 GB) всё равно выполняется целиком; уменьшается число распакованных и используемых записей.
- `SESSION_STEPS = 25` ограничивает число новых обновлений **в одной сессии**, сохраняя полное расписание обучения. Это не ограничение всего эксперимента. `0` снимает ограничение по шагам.
- `SESSION_HOURS = 8` запрашивает завершение после ближайшего полного обновления. Одно обновление может быть долгим; установите запас относительно лимита вашей сессии Kaggle.
- `ACTION = "all"` последовательно выполняет Stage 1 → сохранение пакетов → Stage 2 → оценку. При достижении границы сессии цепочка останавливается. Следующий запуск продолжает незавершённый этап.
- `PAIR = "qwen0.8b_to_qwen0.8b"` — меньшая пара, тоже исследованная в статье. Это отдельный эксперимент; меняйте `WORKDIR` вместе с парой.

Архитектура и функция потерь сохраняются: пакет строится только из контекста, без вопроса; learned queries и cross-attention сжимают его в 32 слота ширины 256 на пару слоёв; gates зависят только от prompt positions; обе LLM заморожены; Stage 1 обучает весь интерфейс, Stage 2 фиксирует compressors и обучает readers/gates. Loss усредняется по всем ответным токенам глобального обновления, а не по средним loss микробатчей. Глобальный batch 384 сохраняется накоплением градиентов. Тексты не обрезаются молча. Пакеты сохраняются в BF16 и дедуплицируются по контексту.

Численная конфигурация отличается от H100 в статье: T4/P100 используют FP16 для LLM, FP32 для обучаемого интерфейса и AdamW, GradScaler для градиентов. Опубликованный код авторов также считает интерфейс в FP32. Поэтому это совпадение логики метода, а не обещание побитового воспроизведения или одинаковой точности. При переполнении runtime останавливается с ошибкой. Квантизация, LoRA и автоматическое уменьшение архитектуры не используются.

Полный эксперимент очень большой. Корпус Stage 1 занимает около 2,8 GB только в сжатом виде. Один стандартный пакет занимает 384 KiB; кэш всех уникальных контекстов может занимать десятки GB. Runtime проверяет свободное место перед кодированием. Две T4 работают через data parallel: память карт не складывается, каждая карта хранит свою копию моделей. Даже с микробатчем 1 фактическая доступность памяти и скорость зависят от среды. Полное обучение потребует многих сессий и не гарантированно помещается в квоты Kaggle.

Для продолжения сохраните notebook outputs и подключите их к следующей сессии как Input. Укажите `RESTORE` на прежнюю папку `latcomm`, содержащую `manifest.json` и `outputs`. Runtime сохраняет bridge, optimizer, scheduler, scaler, RNG и позицию в данных на границе обновления. Он проверяет идентичность данных, моделей, кода и настроек; число GPU и микробатч при точном продолжении менять нельзя. Сохраняются два последних промежуточных checkpoints и финальный.

Каталоги `/kaggle/temp` не сохраняются в output. В ноутбуке туда направлены скачанные модели и JSONL, чтобы не заполнять постоянный output. Данные можно скачать заново с закреплённой ревизии. Пакеты по умолчанию направлены в `WORKDIR/packets`; для полного запуска потребуется достаточно диска и отдельное сохранение кэша. Уже готовый кэш можно подключить через `PACKET_CACHE = "/kaggle/input/.../packets"`; его не нужно копировать. Для продолжения незавершённого кодирования кэш должен быть доступен на запись.

В уже клонированном репозитории можно запускать команды напрямую из его корня:

```bash
python kaggle/run.py setup
python kaggle/run.py all --profile full --gpus 2 --session-steps 25
```

Отдельный `kaggle/bootstrap.py` также умеет клонировать форк, устанавливать зависимости и запускать обучение: `python kaggle/bootstrap.py --profile full --gpus 2`. `run.py` использует репозиторий, в котором находится сам, и ничего дополнительно не клонирует.

Если долго нет первого `train` сообщения, используйте `kaggle/LatComm_Diagnostic.ipynb`. Он запускает три микробатча той же пары моделей, без optimizer updates и без начальной оценки на 256 примерах. Данные выбираются из начала, середины и конца группы накопления, чтобы не измерять только самые короткие тексты. Результат `outputs/stage1/benchmark.json` содержит время микробатчей, проверку конечности градиентов и грубую оценку времени обновления. Короткий холодный замер не включает все накладные расходы и возможные повторы FP16 на полном обновлении.

Новая версия выводит фазы encoder/receiver/backward для первого микробатча каждого обновления, прогресс завершённых микробатчей примерно раз в 30 секунд и отдельное предупреждение при FP16 overflow с повтором группы. Если одна операция блокируется, последней останется строка её фазы; периодический отчёт появляется после завершения микробатча.

Для диагностики используется отдельная папка `/kaggle/working/latcomm-benchmark`. Отправьте обновлённые скрипты в форк перед запуском. Существующий `WORKDIR` закрепляет старый SHA, поэтому обновление main само по себе его код не меняет. Строгая проверка resume отклоняет checkpoints от другой версии runtime; диагностика не меняет старые checkpoints. В той же сессии уже скачанные модели и исходные архивы могут быть повторно использованы из прежнего HF cache.

Оценка пишет `results/metrics.json` и `predictions.jsonl`: точность по каждой задаче, общую точность и среднее по наборам. Версии моделей и датасета фиксируются при первом запуске в `manifest.json`; повторное обучение с изменёнными данными требует отдельного `WORKDIR`.

Проверки в комплекте выполняются без скачивания LLM. Полное обучение на реальных GPU Kaggle в этой локальной среде не выполнялось.

Источники: [официальная реализация](https://github.com/huawei-lin/LatComm), [статья](https://arxiv.org/abs/2610.26840), [совместимость CUDA wheels с P100](https://dev-discuss.pytorch.org/t/notice-cuda-12-6-wheels-will-no-longer-be-published-from-pytorch-2-15-drops-maxwell-pascal-volta/3432).
